# Baseball Lab 4: Calculating the probability of winning a game

In today's lab exercises you'll use the retrosheet play-by-play data to calculate the probability that a baseball team will win a game given the current difference in scores and the current inning and number of outs. Once you have calculated these probabilities, we will apply them to specific games to see how the probability of winning can change throughout a game. 

Please complete this notebook by filling in the cells provided. For all problems that you must write explanations and sentences for, please provide your answer in the designated space. 


#### Deadline

This assignment is due **Sunday February 15th at 11pm**. You can turn in the assignment up to 24 hours late for 90% credit (after that, the homework will only be accepted with a Dean's Extension). Directly sharing answers is not okay, but discussing problems with the course staff or with other students is encouraged. Refer to the policies page to learn more about how to learn cooperatively. You should start early so that you have time to get help if you're stuck. If you have questions, please post them to [Ed Discussion](https://edstem.org/us/courses/89102/discussion) (and answer others' questions too). 


## Getting started - downloading the data

In order to complete this lab, it is necessary to download a few files. Please run the code below **only once** to download data needed to complete the lab. To run the code, click in the cell below and press the play button (or press shift-enter). 


In [24]:
# Please run this code once to download the files you will need to complete the homework 

def download_retro_data(year):
    
    import os.path
    retro_file_name = str(year) + "plays.zip"
    if not os.path.isfile(retro_file_name):
        import requests
        retro_url = "https://www.retrosheet.org/downloads/plays/" + retro_file_name
        r = requests.get(retro_url )
        with open(retro_file_name, "wb") as f:
            f.write(r.content)
    else:
        print("File already downloaded, skipping download step.")


download_retro_data(2019)
download_retro_data(1988)
download_retro_data(1955)

File already downloaded, skipping download step.
File already downloaded, skipping download step.
File already downloaded, skipping download step.


# Part 0: Quote and reaction to Astroball chapter 3 (5 points)

Please find an interesting quote from chapter 4 of Astroball and then write a ~one paragraph reaction to the quote below.

*Quote:*  ...

Reaction: ... 

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Part 1. Warm up exercises: Plotting the slugging percentage for all strikes and ball counts 

Before we get started with our main exercise of calculating the probability of winning given the inning and the score, let's do a warm up exercise to practice manipulating and visualizing data by plotting how slugging percentage changes depending on the exact ball and strike count combination. This will help us get familiar with the retrosheet data and how to create and visualize pivot tables.


**Exercise 1.1 (5 points)**  Using the retrosheet play-by-play data from the 2019 season loaded below, please create a DataFrame called `slg_df` that has the slugging percentage separately for *each ball and strike count combination*. In particular, the rows (index) of the `slg_df` DataFrame should contain the number of strikes, and the column (index) should contain the number of balls. Once you have created this DataFrame, please print it out to "show your work".

Hints: 

1. Recall that Slugging Percentage (SLG) is defined as:  `SLG = (single + 2 * double + 3 * triple + 4 * hr) / at_bats`, where `at_bats` is the number of times the batter has batted (i.e., the number of plate appearances that are not walks or hit by pitches). 

2. We did a very similar exercise in class, so looking at the class 5 code could be helpful if you get stuck (although it would be good to be fluent enough to complete this exercise without looking at any external sources). 


In [26]:
# using dtype to avoid a warning about the umplf and umprf columns
retro_data = pd.read_csv("2019plays.zip", compression='zip', dtype={'umplf': "string", 'umprf': "string"})  

retro_data.head()


,gid,event,inning,top_bot,vis_home,site,batteam,pitteam,score_v,score_h,...,pn,umphome,ump1b,ump2b,ump3b,umplf,umprf,date,gametype,pbp
0,OAK201903200,9/L9M+,1,0,0,TOK01,SEA,OAK,0,0,...,1,nelsj901,gibsh902,barkl901,muchm901,<NA>,<NA>,20190320,regular,full
1,OAK201903200,3/F3D,1,0,0,TOK01,SEA,OAK,0,0,...,2,nelsj901,gibsh902,barkl901,muchm901,<NA>,<NA>,20190320,regular,full
2,OAK201903200,S6/L4D+,1,0,0,TOK01,SEA,OAK,0,0,...,3,nelsj901,gibsh902,barkl901,muchm901,<NA>,<NA>,20190320,regular,full
3,OAK201903200,WP.1-2,1,0,0,TOK01,SEA,OAK,0,0,...,4,nelsj901,gibsh902,barkl901,muchm901,<NA>,<NA>,20190320,regular,full
4,OAK201903200,K,1,0,0,TOK01,SEA,OAK,0,0,...,5,nelsj901,gibsh902,barkl901,muchm901,<NA>,<NA>,20190320,regular,full


## 1.2 Visualizing the slugging percentage 

Now that you have created the `slg_df` DataFrame, please visualize the slugging percentage as a function of the ball and strike count as a heatmap.

**Exercise 1.2 (4 points):** Please create a heatmap of the slugging percentage using the seaborn library. The x-axis should be the number of balls, and the y-axis should be the number of strikes. Please also add a color bar to indicate the slugging percentage values. In the answer section report whether the count has a large effect on the slugging percentage. 

**Answer** 





# Part 2: Exploring the probability of winning a game

The first major league baseball game I attended was on July 16th 1988, where the Boston Red Sox played the Kansas City Royals. By the 6th inning, the Red Sox were down 6-0. Amazingly, the Red Sox made a come back and ended up winning the game 7-6 on a walk off home run by [Kevin Romine](https://www.youtube.com/watch?v=vXsg9Ail19k). The goal of this set of exercises is to assess the probability of such a comeback given the score and the inning.


## 2.0 Interpreting what the 'probability of winning' means

In order to estimate the probability of such a comeback, we will create a DataFrame that has estimates of the probability that a home team will win a baseball game given the inning and the score. In particular, the rows of this probability DataFrame will show the inning/number of outs, and the columns will show the score difference (home - away score). We will create this probability DataFrame from retrosheet play-by-play data from the 2019 season which is loaded below.

**Exercise 2.0 (5 points):** To start, write a few sentences about what is meant by the term "probability of the home team winning a game given the inning and score". Also state whether you think using data from the 2019 season is valid for assessing the probability of winning a game that occurred in 1988.


**Answer:** 





In [27]:
# load the retrosheet play-by-play data from 2019

# using dtype to avoid a warning about the umplf and umprf columns
retro_data = pd.read_csv("2019plays.zip", compression='zip', dtype={'umplf': "string", 'umprf': "string"})  

retro_data.head()


,gid,event,inning,top_bot,vis_home,site,batteam,pitteam,score_v,score_h,...,pn,umphome,ump1b,ump2b,ump3b,umplf,umprf,date,gametype,pbp
0,OAK201903200,9/L9M+,1,0,0,TOK01,SEA,OAK,0,0,...,1,nelsj901,gibsh902,barkl901,muchm901,<NA>,<NA>,20190320,regular,full
1,OAK201903200,3/F3D,1,0,0,TOK01,SEA,OAK,0,0,...,2,nelsj901,gibsh902,barkl901,muchm901,<NA>,<NA>,20190320,regular,full
2,OAK201903200,S6/L4D+,1,0,0,TOK01,SEA,OAK,0,0,...,3,nelsj901,gibsh902,barkl901,muchm901,<NA>,<NA>,20190320,regular,full
3,OAK201903200,WP.1-2,1,0,0,TOK01,SEA,OAK,0,0,...,4,nelsj901,gibsh902,barkl901,muchm901,<NA>,<NA>,20190320,regular,full
4,OAK201903200,K,1,0,0,TOK01,SEA,OAK,0,0,...,5,nelsj901,gibsh902,barkl901,muchm901,<NA>,<NA>,20190320,regular,full


## 2.1 Thinking about the steps needed to create the win probability table

As described above, the goal for this exercises is to create a "win probability DataFrame" (called `win_probability_df`) where:

1. The rows are the innings-out combinations written in the form: `inning_num.out_num` For example, the 7th inning with 1 out would be denoted as 7.1

2. The columns are the differences in the runs that that home team has scored minus the number of runs that the visiting team scored. We will have the range of the columns go from -7 to 7. For example, the value of -5 will indicate that the home team is losing by 5 runs. Additionally, the values of -7 will indicate that the home team is losing by 7 **or more** runs, and +7 will indicate the home team is winning by 7 or more runs. 

3. The values in the DataFrame are estimates of the probability that the home team will win the game. 


**Exercise 2.1 (6 points):** When manipulating data to create a specific desired DataFrame, it is very useful to think about the steps that are needed before doing any coding. When trying to figure out these steps, it is useful to think both in the forward direction, starting with the initial data you have, and also in the backward direction, starting with the final table you want to generate. 

Let's think about the steps in the backward direction. In particular, suppose that for the *second to last* step you had a DataFrame called `simplified_df` where each row of the DataFrame corresponded to a plate appearance, and columns of the DataFrame were: 

1. `inn_out_combo`: A column of strings that has the inning.out combination listed for each inning.out in a season.
2. `score_diff`: A column of numbers that has the difference in runs between the home and away team for each inning.out listed.
3. `home_win` :  A column  of boolean values (1's and 0's) that indicated whether the home team won the game for each inning.out listed.

If you had this DataFrame, what DataFrame method could you use to create the win probability DataFrame, and what arguments would you supply to this method? 

Hint: It could be useful to draw a sketch of the `simplified_df` and the `win_probability_df` so you know what you are starting with, and what you are aiming to create. 


**Answer:**









## 2.2 Creating the score_diff column

Let's now start our actual data manipulation in the forward direction where we begin with the `retro_data` DataFrame. In particular, we will try to derive the `simplified_df` described above in exercise 2.2, and then we will apply the final step to calculate the win probability table.

**Exercise 2.2 (5 points)**:  To begin, let's make the data more manageable, by reducing the `retro_data` DataFrame down to only the columns we will need for subsequent analyses and save this reduced DataFrame to the name `retro_data2`.

The columns that we will need are: 

1. An identifier that uniquely identifies each game.
2. The inning number.
3. Whether the play was in the top or bottom of the inning.
4. The home team score at the end of the inning.
5. The away team score at the end of the inning.
6. Whether a given play was a plate appearance (PA) (we will end up ignoring all events that are not plate appearances).
7. The number of outs before the current play.
8. The number of outs after the current play.

Please create the `retro_data2` DataFrame that has only these columns, and then print the first 3 rows of this DataFrame to show you are on the right track. 
Looking at the [retrosheet documentation](https://www.retrosheet.org/downloads/plays.html?utm_source=chatgpt.com) could be useful if you are not sure what the columns mean. 


## 2.3 Creating the score_diff column

Let's now do a simple operation that adds a column to the `retro_data2` DataFrame that has the difference between the home and away score. This will be the `score_diff` column that we will use later to create the win probability table. 

**Exercise 2.3 (3 points)**: Create a DataFrame called `retro_data3` that is a copy of the `retro_data2` DataFrame. Then add a column called `score_diff` to this `retro_data3` DataFrame which has the difference between the home and away score. Please also print the first 5 rows of this DataFrame to show your work. 


## 2.4 Creating the inn_out_combo column

**Exercise 2.4 (5 points)**:  Next let's add the `inn_out_combo` column to the `retro_data3` DataFrame, which has a combination of the inning and the current number of outs (prior to the current play).  The values in this `inn_out_combo` columns should be a decimal number in the form `inning.outs`, where the inning is the current inning value for each row of the retro data, and the outs are the number of outs prior to the current play. For example, if the inning is 7 and the number of outs is 2, then the `inn_out_combo` should be 7.2. 

You can do this by creating a Series which is equal to `inning + outs/10` and adding it to the  `retro_data3` DataFrame  in a column called `inn_out_combo`. Once you have done this, print the first few rows of the `retro_data3` DataFrame to show your work. 



## 2.5 Indicating whether the home team ends up winning 

Let's now start on a sequence of steps that will add a `home_win` column to the retro data. This column will indicate whether the home team ended up winning the game, for each play that occurred in the game (i.e.,  for each row in the retrosheet data, the column will show whether at the end of the game the home team won).

We can do this by first creating a `end_of_game_df` that indicates for each game (`gid`) whether the home team has won. We can then join this `end_of_game_df` DataFrame onto the `retro_data3` DataFrame to indicate *for each play* whether the home team ended up winning the game.

More specifically, we can add the `home_win` column to the `retro_data3` DataFrame using the following steps:

1. Create a DataFrame called `end_of_game_df` that has the last play of every game.

2. Look at the `end_of_game_df` to determine whether the home team has won by looking at 3 cases:

    a. The visiting team is batting and there are 3 outs (home team win)  
    
    b. The home team is batting and there are 3 outs (home team loss)
    
    c. The home team is batting and there are less than 3 outs (walk off home team win)
    
3. Based on the three cases above, create a DataFrame `home_team_win_df` that has two columns which indicate for each game (`gid`) if the home team won (`home_win`).

4. Join the `retro_data3` with the `home_team_win_df` to list for every play in every game (i.e., each inning.out run differential in every game), whether the home team ended up winning the game.

Before continuing on, please make sure these steps make sense to you!


### 2.5.1 Creating the home_win column (part 1)

Let's now start on a sequence of steps that will add a `home_win` column to the retro data. This column will indicate whether the home team ended up winning the game, for each play that occurred in the game (i.e.,  for each row in the retrosheet data, the column will show whether at the end of the game the home team won).

We can do this by first creating a `end_of_game_df` that indicates for each game (`gid`) whether the home team has won. We can then join this `end_of_game_df` DataFrame onto the `retro_data3` DataFrame to indicate *for each play* whether the home team ended up winning the game.

More specifically, we can add the `home_win` column to the `retro_data3` DataFrame using the following steps:

1. Create a DataFrame called `end_of_game_df` that has the last play of every game.

2. Look at the `end_of_game_df` to determine whether the home team has won by looking at 3 cases:

    a. The visiting team is batting and there are 3 outs (home team win)  
    
    b. The home team is batting and there are 3 outs (home team loss)
    
    c. The home team is batting and there are less than 3 outs (walk off home team win)
    
3. Based on the three cases above, create a DataFrame `home_team_win_df` that has two columns which indicate for each game (`gid`) if the home team won (`home_win`).

4. Join the `retro_data3` with the `home_team_win_df` to list for every play in every game (i.e., each inning.out run differential in every game), whether the home team ended up winning the game.

Before continuing on, please make sure these steps make sense to you!


**Exercise 2.5.1 (5 points)**: Please start with the first step listed above by creating the `end_of_game_df` that contains only the last play of every game. As always, print the first 5 rows of this DataFrame to show your work. 

Hint: One way you can do this is by grouping the `retro_data3` by the unique game ID, and then you can use the `.tail(1)` method to get the last play of each game (we are relying here on the fact that the retro data has each play in a game in the order from the first play to the last play, although it would perhaps be safer to sort the data to be sure). 




### 2.5.2 Creating the home_win column (part 2)

Now let's assess the three scenarios for how the game could have ended that are listed in section 2.5.1, step 2 parts a, b, and c. 

**Exercise 2.5.2 (7 points)**: Please create three DataFrames that have information on how game ended by doing the following:

  a. Create a DataFrame `visiting_team_loss` that has the games that ended with three outs and the visiting team was batting. 

  b. Create a DataFrame `home_team_loss` that has the games that ended with three outs and the home team was batting. 

  c. Create a DataFrame called `home_team_walk_off_win` that has the games ended with the home team batting and there were not three outs. 

Then, as a sanity check that you have covered most of the scenarios, print the sum of the number of rows in the `visiting_team_loss`, `home_team_loss` and `home_team_walk_off_win` and show that it is almost equal to the number of rows in the `end_of_game_df` DataFrame. 

Bonus: see if you can figure out why these numbers are not exactly the same. 


### 2.5.3 Creating the home_win column (part 3)

We are getting close to having the `home_win` column. Let's now do a few more steps to add this column to our `retro_data3` DataFrame. 

**Exercise 2.5.3 (7 points)**: As mentioned above, to create the `home_win` column we can convert the `visiting_team_loss`, `home_team_loss` and `home_team_walk_off_win` DataFrames into DataFrames called `visiting_team_loss2`, `home_team_loss2` and `home_team_walk_off_win2` that only have two columns which are:

1. The `gid` unique ID for each game
2.  A new column `home_win` that has a 1 if the home team has won and a 0 if the home team has lost. 

Please go ahead and do this, and then vertically append these DataFrames together to create a `total_home_team_win_df` using the `pd.concat()` function.

Finally, join the `total_home_team_win_df` on to the `retro_data3` DataFrame and save the result in the name `retro_data4`. Please also print the first 3 rows of the `retro_data4` DataFrame to show your work. 


## 2.6 Preparing to create the win probability DataFrame

Now that we have the columns we need, we are very close to being able to produce our win probability table. As a final step before creating the win probability table, let's create a DataFrame called `simplified_df` that has a cleaned up version of the `retro_data4` DataFrame by doing the steps outlined below.


**Exercise 2.6 (8 points)**: Please create the `simplified_df` that is a cleaned version of the `retro_data4` DataFrame by doing the following: 

1. Use only half of the inning when the home team was batting (i.e., where `top_bot` is 1).

2. Use only events that are plate appearances. 

3. Make all games that go to extra innings count as just 10 innings. 

4. Make all score differentials of more than -7 equal to -7

5. Similarly, make all score differentials of more than +7 equal to +7


## 2.7 Creating the win probability DataFrame

We are now ready to create the win probability DataFrame!

**Exercise 2.7 (5 points)**: Please go ahead and apply the transformation you discussed in exercise 2.2 to the `simplified_df` to create the `win_probability_df`. Show the full DataFrame and report in the answer section the percent of the time that a team that is down by 6 runs in the bottom of the 6th inning will end up winning the game.



**Answer**: 


## 2.8 Visualizing the results

Let's also visualize the win probabilities! 

**Exercise 2.8 (5 points)**:  Let's visualize the win probabilities in two ways:

1. Create line plots that show the win probability as a function of inning/out combination by creating a separate line for each score differential ranging from -3 to +3 (i.e., your final plot should have 7 lines on it).  

2. Use seaborn library to create a heat map of the win probabilities. 


In [28]:
%matplotlib inline












In [29]:
import seaborn as sns
sns.set()








## 2.9 Examining how often did each inning/out run differential occur

In order to get a sense of how much we should trust our probability estimates for different inning/out and run differential combinations, it is useful to know how many games went into each probability estimate value. Let's do this now. 

**Exercise 2.9 (6 points)**:  Please create a DataFrame called `win_probability_count_df` that shows the counts for how often each pair of (inning-outs, run differentials) occurred that went into the `win_probability_df`. In the answer section, report the number of games in the 2019 season in which the home team was losing by six runs in the bottom of the sixth inning. Based on this number, should we have much trust in the estimate we came up with in exercise 2.7? 


**Answer**: 





# Part 3: Plotting the win probability for the full Red Sox game on July 16th 1988 

Now that we have created a DataFrame that has the win probabilities for each inning/out and score differential, we can use this to plot the win probabilities after each play in a specific game. In particular, let's do this for the Red Sox Royals game on July 16th 1988!

The code below loads the retrosheet play-by-play data from the 1988 season and creates a DataFrame called `game_of_interest` that has the Red Sox game played on July 16 1988. We will use this data, in conjunction with the `win_probability_df` DataFrame we created above, to plot the win probabilities for each play in the game. 


In [30]:
retro_data_1988 =  pd.read_csv("1988plays.zip", low_memory=False) 

game_of_interest = retro_data_1988[retro_data_1988['gid'] == 'BOS198807160']  

game_of_interest.head()

,gid,event,inning,top_bot,vis_home,site,batteam,pitteam,score_v,score_h,...,pn,umphome,ump1b,ump2b,ump3b,umplf,umprf,date,gametype,pbp
93291,BOS198807160,D9/L9L,1,0,0,BOS07,KCA,BOS,0,0,...,1,cousd901,roe-r901,koscg901,barnl901,NaN,NaN,19880716,regular,full
93292,BOS198807160,9/F9D.2-3,1,0,0,BOS07,KCA,BOS,0,0,...,2,cousd901,roe-r901,koscg901,barnl901,NaN,NaN,19880716,regular,full
93293,BOS198807160,W,1,0,0,BOS07,KCA,BOS,0,0,...,3,cousd901,roe-r901,koscg901,barnl901,NaN,NaN,19880716,regular,full
93294,BOS198807160,8/SF/F8.3-H,1,0,0,BOS07,KCA,BOS,0,0,...,4,cousd901,roe-r901,koscg901,barnl901,NaN,NaN,19880716,regular,full
93295,BOS198807160,D8/F78.1-H,1,0,0,BOS07,KCA,BOS,1,0,...,5,cousd901,roe-r901,koscg901,barnl901,NaN,NaN,19880716,regular,full


## 3.1 Modifying the game_of_interest DataFrame

Let's start by modifying the `game_of_interest` DataFrame so that it only has the columns we will need for the subsequent steps. 

**Exercise 3.1 (6 points)**: Please modify the `game_of_interest` DataFrame so that it only has the following columns: 

1. The current inning
2. The number of outs before the current play
3. The home team score at the end of the inning
4. The away team score at the end of the inning

Then, create a new column called `inn_out_combo` that has the inning.outs combination as a double (e.g., 1.0 for the first inning with no outs, 2.1 for the second inning with one out, etc.). Finally, create a column called `score_diff` that has the difference between the home team score and the away team score. Please print the first 5 rows of this DataFrame to show your work. 


## 3.2 Converting the win probability DataFrame to long format

Let's now create a long format version of the `win_probability_df` DataFrame that we can use to join onto the `game_of_interest` DataFrame. This DataFrame will have the same information as `win_probability_df`, but in a long format where each row corresponds to a unique inning/out and score differential combination. Having data in this format will allow us to get the win probabilities for each inning/out and score differential situation in the game. 

**Exercise 3.2 (6 points)**: Please create a DataFrame called `win_probability_long_df` that has three columns which are: 

1. `score_diff`: The score differential (home - away) for each inning/out combination.
2. `inn_out_combo`: The inning.outs combination for each inning/out combination.
3. `home_win`: The home win probabilities for each inning/out combination. 

As always, print the first 5 rows of this DataFrame to show your work. 

Hint: One way to do this is to first reset the index of the `win_probability_df` so that `inn_out_combo` becomes a column, and then you can use the `pd.melt()` function to convert the DataFrame into a long format DataFrame. The `id_vars` argument should be set to `inn_out_combo`, the `var_name` argument should be set to `score_diff`, and the `value_name` argument should be set to `home_win`. 


##  3.3 Plotting the win probabilities for the game

Let's now join the `win_probability_long_df` DataFrame onto the `game_of_interest` DataFrame to get the win probabilities for each inning/out and score differential situation in the game. Then we can plot the win probabilities for each play in the game. 

**Exercise 3.3 (6 points)**: Please do the following: 

1. Use the `.merge()` method to do a left join of the `win_probability_long_df` DataFrame onto the `game_of_interest` DataFrame using the `inn_out_combo` and `score_diff` columns as the keys (i.e., `on=['inn_out_combo', 'score_diff']`). Save the result in the name `game_of_interest2`. 

2. Plot the win probabilities for each play in the game using the `plt.plot()` function. The x-axis should the successive plays in the game (i.e., you don't need to set anything for the x-values when plotting), and the y-axis should be the probability that the home team will win. 


# 3.4 Writing a function that will plot the win probabilities for any game

Finally, let's write a function called `plot_win_prob_trajectory(game_of_interest)` that can plot the win probabilities for any game.

**Exercise 3.4 (7 points)**: Please write a function called `plot_win_prob_trajectory(game_of_interest)` that takes a DataFrame, called `game_of_interest` from a retrosheet play-by-play game (i.e., in the same format as the `game_of_interest` DataFrame used above), and produces a plot of the win probabilities for each play in the game. Then apply this function to a Red Sox Yankees game my father went to on his 10th birthday, which was May 7th, 1955. In the answer section, report whether my father happy with how the game turned out? 


In [31]:

def plot_win_prob_trajectory(game_of_interest):

   ...












# plot the game my father went to when the yankees came back (on his 10th birthday)
retro_data_1955 = pd.read_csv("1955plays.zip", low_memory = False)
redsox_yankees_1955 = retro_data_1955.query("gid == 'BOS195505070'")  

plot_win_prob_trajectory(redsox_yankees_1955)


**Answer** 






# 4. Reflection (3 points)

Please fill out the lab 4 reflection on Canvas to let us know how this lab, and the class overall, is going for you. 



# 5.  Submitting your work

Once you're finished filling in and running all cells, you should submit your assignment as a pdf on Gradescope. You can access Gradescope through Canvas on the left-side of the class home page. The problems in each lab assignment are numbered. When submitting on Gradescope, please **make sure to select the correct pages of your pdf that correspond to each problem**. Failure to mark pages correctly **will result in points being deducted** from your score.

To convert this Jupyter notebook document to a pdf please run the code in the cell below. This should produce a document called `lab_02.pdf` which should appear in the files tab on the left (you might need to refresh the files tab to see this file). You can then right click on this file (command click on a mac), to download this pdf document which you can upload to Gradescope. 

Please be sure to check that all the code and output are visible before submitting your pdf to Gradescope as **points will be deducted for missing code and output that is not visible** since we will not be able to grade this.

In [32]:
%%capture

!quarto render lab_04.ipynb --cache-refresh --to pdf 

#### Alternative submission instructions

If converting your Jupyter notebook to a pdf using the command in the cell above does not work, an alternative way to convert your Jupyter notebook is:

1.  Go to "File" at the top-left of your Jupyter Notebook
2.  Under "Download as" (or "Save and Export Notebook As...") and select "HTML (.html)"
3.  After the .html has downloaded, open it and then select "File" and "Print" (note you will not actually be printing)
4.  From the print window, select the option to save as a .pdf
